In [0]:
#Importación de librerías y configuración inicial
import requests, pandas as pd
from pyspark.sql import functions as F

#El endpoint público de la API de Binance para obtener velas o datos históricos de precios (klines)
URL = "https://data-api.binance.vision/api/v3/klines"
#lista con las criptomonedas que se van a consultar: Bitcoin, Ethereum y Solana (en su par con USDT)
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]
#Los nombres oficiales que Binance asigna a las columnas de la respuesta JSON (tiempo de apertura, precio de apertura, máximo, mínimo, cierre, volumen, etc.)
COLS = ["open_time","open","high","low","close","volume","close_time","quote_volume","trades","tb_base","tb_quote","ignore"]

#Extracción de datos (Consumo de la API)
"""Bucle for: Recorre cada criptomoneda de la lista SYMBOLS.

Petición HTTP (requests.get): Pide a Binance las últimas 1000 velas de 1 hora (interval: "1h") para ese símbolo.

Control de errores (r.raise_for_status()): Lanza una excepción si la API falla o devuelve un código de error (como un 404 o 500).

Procesamiento con Pandas: Convierte la respuesta JSON en un DataFrame de Pandas usando los nombres de columnas definidos en COLS, le añade una columna nueva indicando el símbolo (BTCUSDT, etc.) y lo guarda en una lista acumuladora (frames)"""
frames = []
for symbol in SYMBOLS:
    r = requests.get(URL, params={"symbol": symbol, "interval": "1h", "limit": 1000}, timeout=10)
    r.raise_for_status()
    df = pd.DataFrame(r.json(), columns=COLS)
    df["symbol"] = symbol
    frames.append(df)

#Consolidación y conversión a Spark DataFrame
"""pd.concat y .astype(str): Une los DataFrames de las tres criptomonedas en uno solo y convierte todos los campos temporalmente a texto (str) para evitar conflictos de tipos al pasarlos a Spark.

spark.createDataFrame(raw): Transforma el DataFrame de Pandas en un DataFrame distribuido de Apache Spark, permitiendo procesar los datos a escala en Databricks.

Enriquecimiento:

Añade la columna ingested_at con la marca de tiempo exacta del momento en que se cargaron los datos.

Añade la columna source con el valor fijo "binance_api" para saber de dónde provienen (trazabilidad)."""
raw = pd.concat(frames, ignore_index=True).astype(str)
sdf = (spark.createDataFrame(raw)
       .withColumn("ingested_at", F.current_timestamp())
       .withColumn("source", F.lit("binance_api")))

#Escritura en la Capa Bronze       
sdf.write.mode("append").saveAsTable("workspace.bronze.binance_klines_1h")

In [0]:
%sql
SELECT symbol, COUNT(*) 
FROM bronze.binance_klines_1h
GROUP BY symbol